# T1 — `T.Oc38k.roles.e6 → Oc`: the same model, trained only where the temperature is written down

B3's ablation, and the answer to a question its numbers cannot settle. Temperature is
populated in 27.5% of ORD's condition rows, so `?` is the loss-minimizing output and the
model learns to abstain: 85.6% of top-1 generations carry no number at all, though the ones
that do have a median error of 0 °C. Forcing the choice at decode time already lifts top-1
from 6.1% to 27.9% without retraining, which shows the number was in the beam list all along.

This run removes the incentive in the data rather than in the decoder: identical fields,
format, base and hyperparameters, but only the 38,191 training rows (2,098 validation) whose
temperature is recorded. Everything else about it matches B3, so the pair isolates one factor.

**What it costs.** Solvent is populated in 89.5% of the full set and reagents in 79.9%, so
those two fields lose 3.6x their data here. If they hold up, one model serves every field; if
they drop, the temperature slot needs a specialist and the cascade gains a third model.
That trade is the result, not a side effect.

**On its own this is one point, not a curve.** Model 1 gained 7 points going from 57k to 147k
ORD reactions, so 38k is very likely still in the region where data helps. The larger ORD
pool is being extracted in parallel on CPU; the second point comes from it.

**Data:** `kuzmenkoiryna/retro-planner-ord-conditions-temp` — the temperature-bearing subset
of the same roles-split corpus, with the same untouched 5,687-record clean test.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

# The roles split must have travelled with the data, not been re-derived here.
sample = json.loads(open(test_file).readline())
assert "reagents" in sample and "full_reactants_smiles" in sample, "dataset is the pre-roles one"
print("\ninput   :", sample["reactants_smiles"])
print("reagents:", sample["reagents"])
print("as ORD wrote it:", sample["full_reactants_smiles"])

base_model = "t5-small"   # set from B2: wins solvent and catalyst at p<=0.0001
learning_rate = 5e-4
condition_fields = "reagents,solvent,catalyst,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_temp"
time_budget_minutes = 220

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Batch 32 x 1, the script's default: t5-small fits without the accumulation split that
# a 220M model needs on a T4. Effective batch stays 32, matching B2.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_temp \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --max-source-length 256 \
    --max-target-length 256 \
    --learning-rate {learning_rate} \
    --num-train-epochs 6 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
!grep -E "new character token|Train examples|condition field" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
marker = json.load(open(f"{output_dir}/final/conditions_format.json"))
print("format marker:", marker)
assert marker["fields"] == condition_fields.split(","), "marker disagrees with the requested fields"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
best_epoch = min(points, key=lambda p: p[1])[0]
print(f"  best {state.get('best_metric')} at epoch {best_epoch:.2f} of {points[-1][0]:.2f} reached")

In [ ]:
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 32 --device cuda \
    --max-source-length 256 --max-target-length 256 \
    --force-numeric \
    --output "/kaggle/working/T1_conditions_temp38k_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/T1_conditions_temp38k_clean_topk.json"))
summary = data["summary"]
print(json.dumps(summary, indent=2))
print("per-record entries kept for a paired test:", len(data["records"]))

# The criterion, applied as written above rather than reinterpreted now.
reagents_ok = (summary.get("reagents_exact_match_top5") or 0) >= 0.30
print(f"\nreagents strict top-5 = {summary.get('reagents_exact_match_top5')} -> "
      f"{'meets' if reagents_ok else 'misses'} the 30% bar set before the run")
for field in ("solvent", "catalyst"):
    print(f"  {field} strict top-5 = {summary.get(field + '_exact_match_top5')} (compare with B2)")